# 第103章 多分类与Softmax

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 18 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 梯度提升与加法模型  →  **本章任务：** 多分类与Softmax  →  **下一步：** 混淆矩阵与分类指标
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：生活中的很多判断不是非此即彼的"是/否"二选一，而是要在好几类里挑一个——比如把用户评论归类到"满意/中立/不满"，或是判断一封邮件属于"垃圾/促销/普通/重要"。Softmax 把几个类别的原始得分压成一组加起来正好等于 1 的概率，让我们一眼看出哪一类最可能，也方便用交叉熵这类指标评估模型。本章用葡萄酒数据集，把"多分类、Softmax 概率、宏平均与加权平均"这些概念落到能运行的代码上。


## 本章目标

学完本章，你将能够：

- **理解**：理解「多分类与Softmax」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「多分类与Softmax」的关键输出指标。
- **迁移**：能把「多分类与Softmax」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：当类别不止两类，我们要给每个样本一套“属于各类的概率分布”。Softmax 就是把一组得分拧成“加起来等于 1 的概率”的开关——某类占 70%、另一类又占 60%，那一定算错了。多分类还带来一个选择：对每个类别一视同仁（macro），还是按样本量分配权重（weighted）。


- Softmax：\(P(y=k|x)=e^{z_k}/\sum_j e^{z_j}\)
- 每行类别概率之和为 1（打个比方：把“是哪个类别”变成候选们的“得票比例”——合计一定是100%；要是某类占70%、另一类又占60%，那肯定算错了。）
- macro 对每个类别等权
- weighted 按类别样本量加权


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | 只报告总体准确率 |
| 模型、公式与诊断 | `model.predict_proba()`、`model.predict()`、`pd.DataFrame()`、`report.round()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 将类别编号解释成连续数值 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-103 -->
### 数学推导｜Softmax 多分类概率

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜模型为每个类别给出得分。** 类别 $k$ 的 logit 为 $z_k$。

**第 2 步｜指数化使权重为正，再归一化。** 

$$
p_k=\frac{e^{z_k}}{\sum_je^{z_j}},
\qquad
\sum_kp_k=1
$$

**第 3 步｜用真实类别选择对应概率。** one-hot 标签中只有真实类别 $y$ 的 $y_k=1$，所以交叉熵 $-\sum_ky_k\log p_k=-\log p_y$。真实类别概率越小，惩罚越大。

**把上面的关系收束为本章计算式：**

$$
P(y=k\mid x)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}},\qquad L=-\sum_{k=1}^{K}y_k\log p_k
$$

**符号解释：** $z_k$ 是类别 $k$ 的得分，交叉熵惩罚真实类别概率过低。

**代码对应：** 用 `predict_proba` 检查每行概率和为 1，并查看按类别的错误。

**使用边界：** 总体准确率可能掩盖少数类别；需要宏平均指标和混淆矩阵。


In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_wine(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=92
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(
    X_train, y_train
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：上面示例用默认 `test_size=0.25`（代码里没有显式写）把葡萄酒数据切成了训练集和测试集。请试着在 `train_test_split` 里显式加上 `test_size=0.20`，重新切分后观察训练集、测试集的行数怎么变。想一想：训练集变大、测试集变小时，每一类样本被留在训练集里的会更"多"还是更"少"？在类别较少时，这样的切分是不是让每一类都更容易在模型里被"看到"？


In [ ]:
try:
    pass
    # 请在下方填写代码# 练习：在 train_test_split 里显式加上
    # test_size=0.20，重新切分后核对训练/测试行数。import pandas as pdfrom
    # sklearn.model_selection import train_test_split# X、y 已在上面的示例中定义，可直接复用#
    # TODO: 把 test_size 留空位改成 0.20，stratify 与 random_state 保持不变X_train2,
    # X_test2, y_train2, y_test2 = train_test_split(    X, y, test_size=____,
    # stratify=y, random_state=92)print("训练集行数：", len(X_train2), " 测试集行数：",
    # len(X_test2))print("默认(0.25)时：训练", len(X_train), " 测试", len(X_test))#

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, log_loss

prob = model.predict_proba(X_test)
pred = model.predict(X_test)
print("概率行和:", prob[:3].sum(axis=1).round(6))
print("多分类log loss:", round(log_loss(y_test, prob), 3))
report = pd.DataFrame(
    classification_report(
        y_test, pred, target_names=data.target_names, output_dict=True
    )
).T
display(report.round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
_demo_model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(_demo_model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, _demo_model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = _demo_model.predict(X_changed)
print("原始前2个预测：", np.round(_demo_model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(changed_prediction[:2] - _demo_model.predict(X[:2]), 2),
)


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 只报告总体准确率
- 将类别编号解释成连续数值
- 忽略少数类别召回率
- 把 OvR 或 Softmax 的系数直接当成因果效应


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 103.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 103.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 103.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

处理多分类问题，理解 Softmax 概率、宏平均与加权平均指标。


### 你已经掌握

- 训练多分类逻辑回归
- 解释 Softmax 概率
- 读取每类精确率与召回率
- 区分 macro、micro 和 weighted 平均


### 需要注意

- 只报告总体准确率
- 将类别编号解释成连续数值
- 忽略少数类别召回率
- 把 OvR 或 Softmax 的系数直接当成因果效应


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案：显式设置 test_size=0.20，重新切分并核对总体与每一类的行数import pandas as pdfrom
# sklearn.model_selection import train_test_split# X、y 已在上面的示例中定义X_train2,
# X_test2, y_train2, y_test2 = train_test_split(    X, y, test_size=0.20,
# stratify=y, random_state=92)train_n = len(X_train2)test_n =


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
from sklearn.metrics import f1_score

practice_scores = {
    avg: f1_score(y_test, pred, average=avg)
    for avg in ["macro", "micro", "weighted"]
}
print({k: round(v, 3) for k, v in practice_scores.items()})
